# 🩺 第二十三天 · 正式评测 48 题 + 评测报告（异源评测版）

**今天目标（约 2.5 小时）**：用定稿版 `eval_questions_v2.csv` 对**通义千问**跑完全部 48 题，打分、分维度分析，写《评测报告》。

**今天的评测设计（你的改进）**：
- **评测对象 = 通义千问**（与判分助手异源，避免同族幻觉互相掩护）
- **答案对错** = 你的教材/真题答案键（客观对照，你已终审）
- **依据质量** = 异源助手查证式初判 → **你人工终审**，有出入以你为准

**已就绪**：
- `正式评测提示词.txt`：48 题全量（8 批 × 6 题，强化提示词：要求具体理由 + 出处）；
- `eval_records.csv`：48 行空白记录表（旧 pilot 的 DeepSeek 10 题已另存为 `pilot_deepseek_10.csv`，留作跨模型对照）。

> ⚠️ 先 **Kernel → Restart Kernel**。

## 第 1 步 · 用通义千问跑完 48 题（约 1.5 小时）

1. 打开 **tongyi.com** → 新建对话；
2. 打开 `正式评测提示词.txt`，逐批整段复制粘贴（共 **8 批 × 6 题**）；
3. 每批跑完后，把通义的回答 + 「参考资料」来源列表**一起贴回 DSH 聊天窗口**；
4. DSH 助手逐题给出「依据质量初判」（正确/无依据/错误/待查 + 查证理由）；
5. **你复核判定表**，有出入的按你的教材终审；把终审结果填进 `eval_records.csv` 的「模型答案」和「依据质量」两列：
   - 依据质量三档：`正确`（有实质理由+真实出处）/ `无依据`（只报书名/空洞）/ `错误`（理由错误或出处编造）；
   - 填表用 Excel 直接打开 `eval_records.csv`（记得保存），比 Jupyter 方便。

## 第 2 步 · 打分与分析（检查点）

填完后运行下面单元格：算总分、分维度得分、准确率、依据质量分布。

In [ ]:
import pandas as pd

rec = pd.read_csv("eval_records.csv")
rec["正确性得分"] = (rec["标准答案"] == rec["模型答案"]) * 6
rec["依据得分"] = rec["依据质量"].map({"正确": 4, "无依据": 2, "错误": 0})
rec["总分"] = rec["正确性得分"] + rec["依据得分"]
rec.to_csv("eval_records.csv", index=False, encoding="utf-8-sig")

print("=== 总体 ===")
print(f"题数: {len(rec)} | 平均总分: {rec['总分'].mean():.2f}/10")
print(f"准确率: {(rec['正确性得分']>0).mean()*100:.1f}%")
print(f"依据质量分布: {rec['依据质量'].value_counts().to_dict()}")
print()
print("=== 分维度 ===")
print(rec.groupby("维度").agg(准确率=("正确性得分", lambda s: (s>0).mean()*100),
                                平均总分=("总分", "mean"),
                                答错数=("正确性得分", lambda s: (s==0).sum())).round(2))

### 第 2.5 步 · 挑出答错的题做错误分析（练习）

运行：列出所有答错或依据有问题的题。

In [ ]:
rec = pd.read_csv("eval_records.csv")
bad = rec[(rec["正确性得分"] == 0) | (rec["依据质量"] != "正确")]
print(f"问题题数: {len(bad)}")
print(bad[["维度", "维度内题号", "标准答案", "模型答案", "依据质量"]])

## 第 3 步 · 写《评测报告》（核心交付）

在下方 markdown 按模板写报告（把数字替换成你的结果）：

```markdown
# 《医学大模型基础能力评测集 v2》评测报告

## 一、评测对象与方法
对象：通义千问（tongyi.com）
评测集：48 道医学单选题（诊断/用药/指南/伦理 各12题）
打分标准：正确性 6 分 + 依据质量 4 分（满分10分）
答案键质量：双模型交叉验证 + 人工终审（44题一致，4题分歧已裁决）
依据判定：异源助手查证式初判 + 医学专业学生人工终审（防同源幻觉设计）

## 二、总体结果
准确率：___% | 平均总分：___/10
依据质量分布：正确___题 / 无依据___题 / 错误___题

## 三、分维度结果
（贴上面的表格）

## 四、主要发现
1. （哪个维度最强/最弱，为什么）
2. （依据质量暴露了什么问题）
3. （错误分析：模型错在哪类题上）

## 五、局限与改进
1. （本次只测了单一模型，下一步可加 DeepSeek/文心做跨模型对比）
2. （依据判定依赖人工终审，样本量有限等）
```

### 我的评测报告

（按上面模板写在这里）

## ✅ D23 完成标准（打钩）

- [ ] 48 题全部有模型答案和依据质量
- [ ] 依据判定表经本人复核（分歧处以教材终审）
- [ ] 打分脚本跑通（总分/分维度/准确率）
- [ ] 错误分析跑通
- [ ] 评测报告写完
- [ ] 保存（**Cmd + S**）

> 完成后喊我，我把报告帮你润色并推 GitHub——你的核心作品正式上线。